In [0]:
# Live 4 (O) — Orquestração
# Notebook: 12_silver_vendas_pedidos
# Objetivo: criar a Silver no nível de pedido (agregado) a partir dos itens.

from pyspark.sql import functions as F

dbutils.widgets.text("tgt_schema", "aulas_ao_vivo.live_20260323")
TGT_SCHEMA = dbutils.widgets.get("tgt_schema")

TBL_SILVER_ITENS = f"{TGT_SCHEMA}.silver_vendas_itens_pedido"
TBL_SILVER_PEDIDOS = f"{TGT_SCHEMA}.silver_vendas_pedidos"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TGT_SCHEMA}")

itens = spark.table(TBL_SILVER_ITENS)

pedidos_silver = (
    itens
    .groupBy("id_pedido")
    .agg(
        F.min("data_pedido").alias("data_pedido"),
        F.first("dt", ignorenulls=True).alias("dt"),
        F.first("mes_referencia", ignorenulls=True).alias("mes_referencia"),
        F.first("uf", ignorenulls=True).alias("uf"),
        F.first("status", ignorenulls=True).alias("status"),
        F.sum("quantidade").alias("quantidade_total"),
        F.round(F.sum("valor_total"), 2).alias("valor_total_pedido"),
        F.count("produto").alias("qtd_itens_pedido")
    )
)

pedidos_silver.write.mode("overwrite").format("delta").saveAsTable(TBL_SILVER_PEDIDOS)

out = spark.table(TBL_SILVER_PEDIDOS)
out.printSchema()
display(out.orderBy(F.col("data_pedido").asc()).limit(50))
